In [ ]:
param_grid = {
    'learning_rate': [1e-4, 1e-3],
    'batch_size': [32, 64],
    'dropout_rate': [0.25, 0.5],
    'model_architecture': ['ResNet50', 'MobileNetV2', 'EfficientNetB0']
}

print("Updated hyperparameter grid defined:")
print(param_grid)

In [ ]:
def train_and_evaluate_model(learning_rate, batch_size, dropout_rate, model_architecture):
    # 2. Recreate the model architecture based on model_architecture parameter
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.20),
        layers.RandomBrightness(0.20),
    ], name="aug")

    def build_model(img_size: int, num_classes: int, dropout_rate: float, architecture: str) -> tf.keras.Model:
        inputs = keras.Input(shape=(img_size, img_size, 3), name="image", dtype="float32")
        x = data_augmentation(inputs)

        if architecture == 'ResNet50':
            base = keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=(img_size, img_size, 3))
            preprocess_input = keras.applications.resnet.preprocess_input
        elif architecture == 'MobileNetV2':
            base = keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(img_size, img_size, 3))
            preprocess_input = keras.applications.mobilenet_v2.preprocess_input
        elif architecture == 'EfficientNetB0':
            base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(img_size, img_size, 3))
            preprocess_input = keras.applications.efficientnet.preprocess_input
        else:
            raise ValueError(f"Unsupported model architecture: {architecture}")

        x = preprocess_input(x)
        base.trainable = False
        x = base(x, training=False)

        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(dropout_rate)(x)
        x = layers.Dense(256, activation="relu")(x)
        x = layers.Dropout(dropout_rate)(x)
        outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

        model = keras.Model(inputs, outputs, name=f"{architecture}_profilepic_classifier")
        return model

    model = build_model(IMG_SIZE, len(CLASS_NAMES), dropout_rate, model_architecture)

    # 3. Compile the model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # 4. Recreate the tf.data datasets using the provided batch_size
    def make_dataset(paths, labels, batch_size, training=False):
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(buffer_size=min(10000, len(paths)), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
        ds = ds.cache()
        ds = ds.batch(batch_size, drop_remainder=False)
        ds = ds.prefetch(AUTOTUNE)
        return ds

    ds_train = make_dataset(train_paths, train_labels, batch_size, training=True)
    ds_val   = make_dataset(val_paths,   val_labels,   batch_size, training=False)
    ds_test  = make_dataset(test_paths,  test_labels,  batch_size, training=False)


    # 5. Train the model
    class SaveBest(keras.callbacks.Callback):
        def __init__(self, filepath, monitor='val_loss', mode='min'):
            super().__init__()
            self.filepath = filepath
            self.monitor = monitor
            self.mode = mode
            self.best = float('inf') if mode == 'min' else -float('inf')

        def on_epoch_end(self, epoch, logs=None):
            logs = logs or {}
            value = logs.get(self.monitor)
            if value is None:
                return
            improved = (value < self.best) if self.mode == 'min' else (value > self.best)
            if improved:
                self.best = value
                # Avoid printing during grid search to keep output clean
                # print(f"Saved improved weights to {self.filepath} (epoch {epoch+1}, {self.monitor}={value:.5f})")

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
        SaveBest(f"{model_architecture}_best_{learning_rate}_{batch_size}_{dropout_rate}.weights.h5", monitor="val_loss", mode="min"), # Unique name
    ]

    history = model.fit(
        ds_train,
        validation_data=ds_val,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=0
    )

    # 6. Evaluate the trained model
    test_metrics = model.evaluate(ds_test, return_dict=True, verbose=0)

    # 7. Return evaluation metrics and history
    return test_metrics, history.history

In [ ]:
import itertools
import time

results = []

# Iterate through all combinations of hyperparameters
for lr, bs, dp, arch in itertools.product(
    param_grid['learning_rate'],
    param_grid['batch_size'],
    param_grid['dropout_rate'],
    param_grid['model_architecture']
):
    print(f"Training with architecture={arch}, lr={lr}, bs={bs}, dp={dp}")
    start_time = time.time()

    # Train and evaluate the model with the current hyperparameters
    try:
        test_metrics, history = train_and_evaluate_model(lr, bs, dp, arch)

        # Store the results
        results.append({
            'model_architecture': arch,
            'learning_rate': lr,
            'batch_size': bs,
            'dropout_rate': dp,
            'test_loss': test_metrics['loss'],
            'test_accuracy': test_metrics['accuracy'],
            'history': history # Store history for potential later analysis
        })
    except Exception as e:
        print(f"Error during training for {arch}, lr={lr}, bs={bs}, dp={dp}: {e}")
        results.append({
            'model_architecture': arch,
            'learning_rate': lr,
            'batch_size': bs,
            'dropout_rate': dp,
            'test_loss': None,
            'test_accuracy': None,
            'history': None,
            'error': str(e)
        })

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Completed training for {arch}, lr={lr}, bs={bs}, dp={dp} in {elapsed_time:.2f} seconds.")

print("\nGrid search completed.")

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
print("Hyperparameter and Model Architecture Grid Search Results:")
display(results_df)

In [ ]:
# Find the best performing combination based on test accuracy (highest) and test loss (lowest)
best_combination = results_df.sort_values(by=['test_accuracy', 'test_loss'], ascending=[False, True]).iloc[0]

print("Best performing model and hyperparameter combination:")
display(best_combination)